In [41]:
from dotenv import load_dotenv
load_dotenv()

True

In [42]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    query: str # 사용자 질문
    answer: str # 세율
    tax_base_equation: str # 과세표준 계산 수식 
    tax_deduction: str # 공제액 
    market_ratio: str # 공정시장가액비율
    tax_base: str # 과세표준 계산

graph_builder = StateGraph(AgentState)

In [43]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(model='text-embedding-3-large')

vector_store = Chroma(
    embedding_function=embedding_function,
    collection_name = 'real_estate_tax',
    persist_directory='./real_estate_tax_collection'
)
retriever = vector_store.as_retriever(search_kwargs={'k': 3})

In [44]:
query = '5억짜리 집 1채, 10억짜리 집 1채, 20억짜리 집 1채를 가지고 있을 때 세금을 얼마나 내나요?'

In [45]:
from langchain_openai import ChatOpenAI
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate


rag_prompt = hub.pull('rlm/rag-prompt')
llm = ChatOpenAI(model="gpt-4o")
small_llm = ChatOpenAI(model="gpt-4o-mini")

In [46]:

tax_base_retrieval_chain = (
    {'context': retriever, 'question': RunnablePassthrough()} 
    | rag_prompt 
    | llm 
    | StrOutputParser()
)

tax_base_equation_promt = ChatPromptTemplate.from_messages([
    ('system', '사용자의 질문에서 과세표준을 계산하는 방법을 수식으로 나타내주세요. 부연설명 없이 수식만 리턴해주세요'),
    ('human', '{tax_base_equation_information}')
])

tax_base_equation_chain = (
    {'tax_base_equation_information': RunnablePassthrough()}
    | tax_base_equation_promt
    | llm
    | StrOutputParser()
)

tax_base_chain = {'tax_base_equation_information': tax_base_retrieval_chain} | tax_base_equation_chain


def get_tax_base_equation(state: AgentState):
    tax_base_equation_question = '주택에 대한 종합부동산세 계산시 과세표준을 계산하는 방법을 수식으로 표현해서 알려주세요'
    tax_base_equation = tax_base_equation_chain.invoke(tax_base_equation_question)
    return {'tax_base_equation': tax_base_equation}

In [47]:
get_tax_base_equation({})


{'tax_base_equation': '과세표준 = (공시가격 × 공정시장가액비율) - 기본공제액'}

In [48]:
tax_deduction_chain = (
    {'context': retriever, 'question': RunnablePassthrough()} 
    | rag_prompt 
    | llm 
    | StrOutputParser()
)

def get_tax_deduction(state: AgentState):
    tax_deduction_question = '주택에 대한 종합부동산세 계산시 공제금액을 알려주세요'
    tax_deduction = tax_deduction_chain.invoke(tax_deduction_question)
    return {'tax_deduction': tax_deduction}

In [49]:
get_tax_deduction({})


{'tax_deduction': '주택에 대한 종합부동산세 계산시 1세대 1주택자의 경우 공제금액은 12억 원입니다. 기타 경우에는 9억 원이 공제됩니다. 법인 또는 법인으로 보는 단체는 6억 원의 공제를 받습니다.'}

In [50]:
from langchain_community.tools import TavilySearchResults
from datetime import date

tavily_search_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=True,
    include_images=True,
)

tax_market_ratio_prompt = ChatPromptTemplate.from_messages([
    ('system', f'아래 정보를 기반으로 공정시장 가액비율을 계산해주세요\n\nContext:\n{{context}}'),
    ('human', '{query}')
])

def get_market_ratio(state: AgentState):
    query = f'오늘 날짜:({date.today()})에 해당하는 주택 공시가격 공정시장가액비율은 몇%인가요?'
    context = tavily_search_tool.invoke(query)
    print({'context': context})
    tax_market_ratio_prompt_chain = (
        tax_market_ratio_prompt
        | llm
        | StrOutputParser()
    )
    market_ratio = tax_market_ratio_prompt_chain.invoke({'context': context, 'query': query})
    return {'market_ratio': market_ratio}

In [51]:
get_market_ratio({})

{'context': [{'title': '[2026 공시가격] 李정부, 공시가 손질 착수...내년부터 상승 가능성', 'url': 'https://v.daum.net/v/20260317160802885', 'content': '부동산 업계에서는 공시가격이 재산세와 종합부동산세 등 보유세 산정의 기준이 되는 핵심 지표인 만큼, 공시가격 현실화율이 조정될 경우 보유세 변화는 불가피할 것으로 분석한다.\n\n현재 재산세는 1가구 1주택자를 기준으로 공정시장가액비율이 공시가격 3억원 초과 6억원 이하 구간은 44%, 6억원 초과 구간은 45%가 적용된다. 종합부동산세는 1가구 1주택 기준 기본공제 12억원과 공정시장가액비율 60%가 적용된다.\n\n이에 따라 공시가격이 상승하면 재산세와 종합부동산세 과세표준이 높아지면서 보유세 부담도 함께 증가하게 된다. 특히 고가 주택의 경우, 종합부동산세 과세 대상이 되는 경우가 많아 공시가격 변동에 따른 세금 영향이 상대적으로 크게 나타날 수 있다.\n\n서진형 광운대 부동산법무학과 교수는 "공시가격 현실화율이 90% 수준으로 상향될 경우 세 부담을 느낀 보유자들이 매물을 시장에 내놓으면서 일정 부분 공급 확대 효과를 기대할 수 있다"면서도 "다만 조세 부담이 급격히 늘어나면 임대료로 전가되는 등 부작용이 나타날 수 있는 만큼, 국민 수용성을 고려한 정교한 정책 운용이 필요하다"고 말했다.\n\n이재성 기자 ljs@newsway.co.kr\n\nCopyright © 뉴스웨이. 무단전재, 재배포, AI 학습 이용 금지\n\n### 이 시각 추천뉴스\n\n### 실시간 트렌드\n\n실시간 트렌드 데이터는 안정화 기간 중 베타(Beta) 서비스로 운영하며, 01:00~06:00에는 데이터 수집 규모와 이용률이 낮아 제한적으로 제공합니다.\n\n### 서비스 바로가기 [...] # 뉴스웨이\n\n## 경제\n\n### [2026 공시가격] 李정부, 공시가 손질 착수...내년부터 상승 가능성\n\n이 글자크기로 변경됩니다.\n\n(예시

{'market_ratio': '2026년 현재 주택에 적용되는 공정시장가액비율은 60%입니다.'}

In [ ]:
from langchain_core.prompts import PromptTemplate

tax_base_calculation_prompt = ChatPromptTemplate.from_messages(
    [
        ('system',"""
주어진 내용을 기반으로 과세표준을 계산해주세요

과세표준 계산 공식: {tax_base_equation}
공제금액: {tax_deduction}
공정시장가액비율: {market_ratio}"""),
        ('human', '사용자 주택 공시가격 정보: {query}')
    ]
)

def calculate_tax_base(state: AgentState):
    tax_base_equation = state['tax_base_equation']
    tax_deduction = state['tax_deduction']
    market_ratio = state['market_ratio']
    query = state['query']
    tax_base_calculation_chain = (
        tax_base_calculation_prompt
        | llm
        | StrOutputParser()
    )
    tax_base = tax_base_calculation_chain.invoke({
        'tax_base_equation': tax_base_equation,
        'tax_deduction': tax_deduction,
        'market_ratio': market_ratio,
        'query': query
    })
    print(f'tax_base == {tax_base}')
    return {'tax_base': tax_base}

In [53]:
initial_state = {
    'query': query,
    'tax_base_equation': '과세표준 = (공시가격 × 공정시장가액비율) - 종합부동산세 공제금액',
    'tax_deduction': '주택에 대한 종합부동산세 계산 시 1세대 1주택자의 공제금액은 12억 원입니다. 법인이나 법인으로 보는 단체의 경우는 6억 원, 그 외의 자는 9억 원이 공제됩니다.',
    'market_ratio': '공정시장가액비율은 2024년 기준으로 60%입니다.'
    
}

In [54]:
calculate_tax_base(initial_state)

tax_base == 주어진 정보를 바탕으로 각 주택에 대한 과세표준을 계산해보겠습니다. 

사용자는 총 3채의 집을 가지고 있습니다:
1. 5억 원짜리 집 1채
2. 10억 원짜리 집 1채
3. 20억 원짜리 집 1채

각 주택의 공시가격에 대해 2024년 기준 공정시장가액비율 60%를 적용하여 계산할 수 있습니다. 또한, 1세대 1주택자의 경우 공제금액은 12억 원입니다. 다세대 주택일 경우 공제금액은 각 주택에 대해 적용되지 않으며, 전체 주택의 공시가격 합계를 기준으로 계산해야 합니다.

전체 공시가격 합계:
5억 원 + 10억 원 + 20억 원 = 35억 원

과세표준 계산:
1. 공정시장가액 계산: 
   (35억 원 × 60%) = 21억 원

2. 공제금액: 
   다세대 주택 소유자인 경우 공제금액은 9억 원입니다.

3. 과세표준:
   21억 원 - 9억 원 = 12억 원

따라서, 사용자의 과세표준은 12억 원입니다. 이 과세표준을 바탕으로 정부가 정한 종합부동산세율을 적용하여 세금을 계산하게 됩니다. 세율 정보는 따로 제공되지 않아서 여기서는 과세표준까지만 계산하였습니다.


{'tax_base': '주어진 정보를 바탕으로 각 주택에 대한 과세표준을 계산해보겠습니다. \n\n사용자는 총 3채의 집을 가지고 있습니다:\n1. 5억 원짜리 집 1채\n2. 10억 원짜리 집 1채\n3. 20억 원짜리 집 1채\n\n각 주택의 공시가격에 대해 2024년 기준 공정시장가액비율 60%를 적용하여 계산할 수 있습니다. 또한, 1세대 1주택자의 경우 공제금액은 12억 원입니다. 다세대 주택일 경우 공제금액은 각 주택에 대해 적용되지 않으며, 전체 주택의 공시가격 합계를 기준으로 계산해야 합니다.\n\n전체 공시가격 합계:\n5억 원 + 10억 원 + 20억 원 = 35억 원\n\n과세표준 계산:\n1. 공정시장가액 계산: \n   (35억 원 × 60%) = 21억 원\n\n2. 공제금액: \n   다세대 주택 소유자인 경우 공제금액은 9억 원입니다.\n\n3. 과세표준:\n   21억 원 - 9억 원 = 12억 원\n\n따라서, 사용자의 과세표준은 12억 원입니다. 이 과세표준을 바탕으로 정부가 정한 종합부동산세율을 적용하여 세금을 계산하게 됩니다. 세율 정보는 따로 제공되지 않아서 여기서는 과세표준까지만 계산하였습니다.'}

In [55]:
tax_rate_calculation_prompt = ChatPromptTemplate.from_messages([
    ("system", '''당신은 종합부동산세 계산 전문가입니다. 아래 문서를 참고해서 사용자의 질문에 대한 종합부동산세를 계산해주세요

종합부동산세 세율:{context}'''),
    ("user", '''과세표준과 사용자가 소지한 주택의 수가 아래와 같을 때 종합부동산세를 계산해주세요

과세표준: {tax_base}
주택 수:{query}''')
])


def calculate_tax_rate(state: AgentState):
    query = state['query']
    tax_base = state['tax_base']
    context = retriever.invoke(query)
    tax_rate_chain = (
        tax_rate_calculation_prompt
        | llm
        | StrOutputParser()
    )
    tax_rate = tax_rate_chain.invoke({
        'context': context,
        'tax_base': tax_base,
        'query': query})
    print(f'tax_rate == {tax_rate}')
    return {'answer': tax_rate}

In [56]:
calculate_tax_base(initial_state)

tax_base == 두 가지 경우로 계산이 필요합니다: 1세대 1주택자일 때와 그 외의 경우입니다.

1. 1세대 1주택자:
   - 공제금액: 12억 원

   사용자가 가지고 있는 집들의 총 공시가격:
   - 5억 + 10억 + 20억 = 35억 원

   과세표준 계산:
   - 과세표준 = (공시가격 × 공정시장가액비율) - 공제금액
   - 과세표준 = (35억 × 0.6) - 12억
   - 과세표준 = 21억 - 12억
   - 과세표준 = 9억 원

2. 그 외의 경우 (법인 등):
   - 공제금액: 9억 원

   과세표준 계산:
   - 과세표준 = (35억 × 0.6) - 9억
   - 과세표준 = 21억 - 9억
   - 과세표준 = 12억 원

그러므로, 1세대 1주택자의 경우 과세표준은 9억 원이고, 그 외의 경우에는 12억 원이 됩니다.


{'tax_base': '두 가지 경우로 계산이 필요합니다: 1세대 1주택자일 때와 그 외의 경우입니다.\n\n1. 1세대 1주택자:\n   - 공제금액: 12억 원\n\n   사용자가 가지고 있는 집들의 총 공시가격:\n   - 5억 + 10억 + 20억 = 35억 원\n\n   과세표준 계산:\n   - 과세표준 = (공시가격 × 공정시장가액비율) - 공제금액\n   - 과세표준 = (35억 × 0.6) - 12억\n   - 과세표준 = 21억 - 12억\n   - 과세표준 = 9억 원\n\n2. 그 외의 경우 (법인 등):\n   - 공제금액: 9억 원\n\n   과세표준 계산:\n   - 과세표준 = (35억 × 0.6) - 9억\n   - 과세표준 = 21억 - 9억\n   - 과세표준 = 12억 원\n\n그러므로, 1세대 1주택자의 경우 과세표준은 9억 원이고, 그 외의 경우에는 12억 원이 됩니다.'}